# 🧪 Fase 2: Laboratório de Micro-Engenharia e Modelagem de Especialistas

## Tese de Modelagem
Grandes modelos falham em enxergar peculiaridades regionais. Neste notebook, usaremos as coordenadas do SOM para abrir dois laboratórios de Engenharia de Features:
1. **Hemisfério Norte:** Mercado de alta volatilidade, focado em atrito de preço relativo contra vizinhanças financeiras locais agrupadas por densidade via **DBSCAN**.
2. **Hemisfério Sul:** Mercado de fidelidade familiar estruturado em indicações e tipos de ofertas.
3. **Multi-Quadrantes:** Expansão geométrica em 4 eixos para testar a injeção de equações matemáticas vetorizadas hiper-específicas (Dados vs Voz).

## Output Técnico
Mapeamento completo dos limites comportamentais da base e validação cruzada estruturada via XGBoost regionalizado.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, recall_score, roc_auc_score, precision_score
import joblib

# Carregando a ponte do notebook anterior
df_analise = joblib.load('../data/processed/df_com_coordenadas_som.pkl')
regiao_norte = df_analise[df_analise['Y_som'] >= 20].copy()
regiao_sul = df_analise[df_analise['Y_som'] < 20].copy()

# =========================================================================
# LABORATÓRIO 1: SUB-CLUSTERIZAÇÃO COM DBSCAN NO NORTE (ANATOMIA DE ILHAS)
# =========================================================================
print("=================== LABORATÓRIO: DBSCAN ECOSSISTEMA EXPANDIDO (NORTE) ===================")
todos_servicos = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtectionPlan', 'PremiumTechSupport', 'StreamingTV', 'StreamingMovies', 'StreamingMusic', 'UnlimitedData']
df_statistics_exp = regiao_norte[['CustomerStatus', 'MonthlyCharge'] + todos_servicos].copy()

X_expandido = df_statistics_exp[['MonthlyCharge']].copy()
for col in todos_servicos:
    X_expandido[col] = np.where(df_statistics_exp[col] == 'Yes', 1, 0)

scaler_dbscan_exp = StandardScaler()
X_expandido_scaled = scaler_dbscan_exp.fit_transform(X_expandido)

# Executando o DBSCAN de Alta Resolução espacial
dbscan_expandido = DBSCAN(eps=2.253, min_samples=10)
regiao_norte['Ilha_Expandida'] = dbscan_expandido.fit_predict(X_expandido_scaled)

# Construindo o Relatório de Anatomia das Ilhas para exibição no portfólio
df_analise_exp = X_expandido.copy()
df_analise_exp['Ilha'] = regiao_norte['Ilha_Expandida']
df_analise_exp['Is_Churn'] = np.where(regiao_norte['CustomerStatus'] == 'Churned', 1, 0)

agregacoes = {'MonthlyCharge': ['count', 'mean'], 'Is_Churn': 'mean'}
for col in todos_servicos:
    agregacoes[col] = 'mean'

relatorio_exp = df_analise_exp.groupby('Ilha').agg(agregacoes)
relatorio_exp.columns = ['Total_Clientes', 'Media_Preco', 'Taxa_Churn'] + [f'Adocao_{c}' for c in todos_servicos]

for col in [c for c in relatorio_exp.columns if 'Adocao' in c or 'Taxa' in c]:
    relatorio_exp[col] = (relatorio_exp[col] * 100).round(1).astype(str) + '%'
relatorio_exp['Media_Preco'] = relatorio_exp['Media_Preco'].round(2)

print("\n--- ANATOMIA VISUAL DAS ILHAS DO NORTE (DBSCAN) ---")
print(relatorio_exp.reset_index().to_string(index=False))

# =========================================================================
# LABORATÓRIO 2: ENGENHARIA DA FEATURE AUTORAL PLUS-MINUS
# =========================================================================
print("\n=================== CONSTRUÇÃO DA METODOLOGIA PLUS-MINUS (±) ===================")
mapa_medias_ilhas = regiao_norte.groupby('Ilha_Expandida')['MonthlyCharge'].mean().to_dict()
regiao_norte['Island_Mean_Price'] = regiao_norte['Ilha_Expandida'].map(mapa_medias_ilhas)
regiao_norte['Price_Plus_Minus'] = regiao_norte['MonthlyCharge'] - regiao_norte['Island_Mean_Price']

cushion_suporte = np.where(regiao_norte['PremiumTechSupport'] == 'Yes', -5.0, 0.0)
cushion_protecao = np.where(regiao_norte['DeviceProtectionPlan'] == 'Yes', -3.0, 0.0)
cushion_dados = np.where(regiao_norte['UnlimitedData'] == 'Yes', -1.0, 0.0)
regiao_norte['Service_Cushion'] = cushion_suporte + cushion_protecao + cushion_dados

# Super-feature indexada de custo-benefício relativo
regiao_norte['Island_Plus_Minus_Index'] = regiao_norte['Price_Plus_Minus'] + regiao_norte['Service_Cushion']
regiao_norte['San_Diego_Effect'] = np.where(regiao_norte['City'] == 'San Diego', 1, 0)

print("Média do Plus-Minus por Status Real do Cliente:")
print(regiao_norte.groupby('CustomerStatus')['Island_Plus_Minus_Index'].mean())

# =========================================================================
# LABORATÓRIO 3: TREINAMENTO DO MODELO ESPECIALISTA DO NORTE
# =========================================================================
print("\n=================== ESTEIRA ESPECIALISTA: HEMISFÉRIO NORTE ===================")
vazamentos_e_ruidos = ['CustomerID', 'Country', 'State', 'City', 'ZipCode', 'Latitude', 'Longitude', 'Quarter', 'CustomerStatus', 'ChurnLabel', 'ChurnScore', 'ChurnCategory', 'ChurnReason', 'SatisfactionScore', 'OnlineSecurity', 'X_som', 'Y_som', 'Ilha_Expandida', 'Island_Mean_Price', 'Price_Plus_Minus', 'Service_Cushion']

X_norte_fe = regiao_norte.drop(columns=vazamentos_e_ruidos, errors='ignore')
y_norte_fe = np.where(regiao_norte['CustomerStatus'] == 'Churned', 1, 0)

cols_cat_fe = X_norte_fe.select_dtypes(include=['object', 'category']).columns.tolist()
X_norte_fe_encoded = pd.get_dummies(X_norte_fe, columns=cols_cat_fe, drop_first=True)

X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(X_norte_fe_encoded, y_norte_fe, test_size=0.3, random_state=42, stratify=y_norte_fe)

proporcao_norte = (len(y_train_n) - sum(y_train_n)) / sum(y_train_n)
xgb_norte = XGBClassifier(scale_pos_weight=proporcao_norte, eval_metric='logloss', random_state=42)

grid_norte = GridSearchCV(xgb_norte, {'max_depth': [3, 4], 'learning_rate': [0.01, 0.05], 'n_estimators': [100, 150], 'gamma': [1, 2]}, scoring='recall', cv=5, n_jobs=-1)
grid_norte.fit(X_train_n, y_train_n)

print("\n--- PERFORMANCE DO ESPECIALISTA DO NORTE ---")
best_norte = grid_norte.best_estimator_
y_pred_n = best_norte.predict(X_test_n)
y_prob_n = best_norte.predict_proba(X_test_n)[:, 1]

print(f"Melhores parâmetros: {grid_norte.best_params_}")
print(f"ROC-AUC: {roc_auc_score(y_test_n, y_prob_n)*100:.2f}%")
print(f"RECALL:  {recall_score(y_test_n, y_pred_n)*100:.2f}%")
print(f"PRECISÃO: {precision_score(y_test_n, y_pred_n)*100:.2f}%")

print("\nMatriz de Confusão (Hemisfério Norte):")
cm_norte = confusion_matrix(y_test_n, y_pred_n)
print(f"  [Clientes do Norte que FICARAM previstos corretamente]: {cm_norte[0][0]}")
print(f"  [Clientes do Norte que FICARAM mas gerou Alarme Falso]: {cm_norte[0][1]}")
print(f"  [Clientes do Norte que REALMENTE CANCELARAM e cometeu Erro Crítico]: {cm_norte[1][0]}")
print(f"  [Clientes do Norte que REALMENTE CANCELARAM previstos corretamente]: {cm_norte[1][1]}")

# =========================================================================
# LABORATÓRIO 4: ENGENHARIA DE VULNERABILIDADE E MODELO DO SUL
# =========================================================================
print("\n=================== LABORATÓRIO: OFERTAS + CONTRATOS (SUL) ===================")
df_internet_sul = regiao_sul[regiao_sul['InternetService'] == 'Yes'].copy()
print("--- Cruzamento de Ouro: Oferta vs Tipo de Contrato vs Churn ---")
oferta_contrato = pd.crosstab(df_internet_sul['Offer'], [df_internet_sul['Contract'], df_internet_sul['CustomerStatus']], normalize='index') * 100
print(oferta_contrato.round(1).astype(str) + '%')

# Criando a feature cirúrgica de Ofertas Vulneráveis no Sul
condicao_vulneravel = regiao_sul['Offer'].isin(['Offer E', 'Offer C', 'Offer D']) & (regiao_sul['Contract'] == 'Month-to-month')
regiao_sul['Offer_Contract_Vulnerability'] = np.where(condicao_vulneravel, 1, 0)
regiao_sul['San_Diego_Effect'] = np.where(regiao_sul['City'] == 'San Diego', 1, 0)

print("\n=================== ESTEIRA ESPECIALISTA: HEMISFÉRIO SUL ===================")
X_sul_fe = regiao_sul.drop(columns=[c for c in vazamentos_e_ruidos if c in regiao_sul.columns], errors='ignore')
y_sul_fe = np.where(regiao_sul['CustomerStatus'] == 'Churned', 1, 0)

cols_cat_sul = X_sul_fe.select_dtypes(include=['object', 'category']).columns.tolist()
X_sul_fe_encoded = pd.get_dummies(X_sul_fe, columns=cols_cat_sul, drop_first=True)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_sul_fe_encoded, y_sul_fe, test_size=0.3, random_state=42, stratify=y_sul_fe)

proporcao_sul = (len(y_train_s) - sum(y_train_s)) / sum(y_train_s)
xgb_sul = XGBClassifier(scale_pos_weight=proporcao_sul, eval_metric='logloss', random_state=42)

grid_sul = GridSearchCV(xgb_sul, {'max_depth': [3, 4], 'learning_rate': [0.01, 0.05], 'n_estimators': [100, 150], 'gamma': [1, 2]}, scoring='recall', cv=5, n_jobs=-1)
grid_sul.fit(X_train_s, y_train_s)

print("\n--- PERFORMANCE DO ESPECIALISTA DO SUL ---")
best_sul = grid_sul.best_estimator_
y_pred_s = best_sul.predict(X_test_s)
y_prob_s = best_sul.predict_proba(X_test_s)[:, 1]

print(f"Melhores parâmetros: {grid_sul.best_params_}")
print(f"ROC-AUC: {roc_auc_score(y_test_s, y_prob_s)*100:.2f}%")
print(f"RECALL:  {recall_score(y_test_s, y_pred_s)*100:.2f}%")
print(f"PRECISÃO: {precision_score(y_test_s, y_pred_s)*100:.2f}%")

print("\nMatriz de Confusão (Hemisfério Sul):")
cm_sul = confusion_matrix(y_test_s, y_pred_s)
print(f"  [Clientes do Sul que FICARAM previstos corretamente]: {cm_sul[0][0]}")
print(f"  [Clientes do Sul que FICARAM mas gerou Alarme Falso]: {cm_sul[0][1]}")
print(f"  [Clientes do Sul que REALMENTE CANCELARAM e cometeu Erro Crítico]: {cm_sul[1][0]}")
print(f"  [Clientes do Sul que REALMENTE CANCELARAM previstos corretamente]: {cm_sul[1][1]}")

print("\n[Fim do Laboratório] Todos os dados experimentais e matrizes de confusão foram devidamente gerados.")

=================== LABORATÓRIO: DBSCAN ECOSSISTEMA EXPANDIDO (NORTE) ===================

--- ANATOMIA VISUAL DAS ILHAS DO NORTE (DBSCAN) ---
 Ilha  Total_Clientes  Media_Preco Taxa_Churn Adocao_OnlineSecurity Adocao_OnlineBackup Adocao_DeviceProtectionPlan Adocao_PremiumTechSupport Adocao_StreamingTV Adocao_StreamingMovies Adocao_StreamingMusic Adocao_UnlimitedData
   -1               1        96.30     100.0%                100.0%                0.0%                      100.0%                    100.0%               0.0%                 100.0%                  0.0%                 0.0%
    0             350        75.37      48.0%                  0.0%              100.0%                        0.0%                      0.0%              38.9%                  37.7%                 34.9%                86.3%
    1            1824        51.27      41.5%                  0.0%                0.0%                        0.0%                      0.0%              19.1%                